# Objetivo del Notebook (CRISP-DM Fase 3: Data Preparation)
Este notebook valida la infraestructura del dataset YOLO multiclase:
- Confirmar la estructura de directorios en `dataset/selected/` (`train`, `val`, `test`).
- Validar pasivamente la configuración oficial en `ml/configs/data.yaml` (`nc: 2`, `caries_permanente` y `caries_decidua`).
- Asegurar la integridad de las rutas relativas portables (sin paths absolutos dependientes de máquina).
- Verificar el inventario de 1,957 imágenes intraorales con aislamiento por paciente y controles sanos.

In [1]:
# Importar librerías básicas
from pathlib import Path
import re
import yaml

In [2]:
# Definir rutas base relativas y portables (compatibles con Windows, macOS M1 Pro y Linux)
PROJECT_ROOT = Path("..").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset" / "selected"
DATA_YAML_PATH = PROJECT_ROOT / "ml" / "configs" / "data.yaml"

print(f"Ruta raíz del proyecto: {PROJECT_ROOT.name}")
print(f"Directorio del dataset: {DATASET_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Ruta config YOLO:       {DATA_YAML_PATH.relative_to(PROJECT_ROOT).as_posix()}")

Ruta raíz del proyecto: dental-caries
Directorio del dataset: dataset/selected
Ruta config YOLO:       ml/configs/data.yaml


In [3]:
# Verificar estructura canónica del dataset YOLO
expected_dirs = [
    DATASET_DIR / "images" / "train",
    DATASET_DIR / "images" / "val",
    DATASET_DIR / "images" / "test",
    DATASET_DIR / "labels" / "train",
    DATASET_DIR / "labels" / "val",
    DATASET_DIR / "labels" / "test",
]

for d in expected_dirs:
    status = "OK" if d.exists() else "FALTA"
    print(f"[{status}] {d.relative_to(PROJECT_ROOT).as_posix()}")

[OK] dataset/selected/images/train
[OK] dataset/selected/images/val
[OK] dataset/selected/images/test
[OK] dataset/selected/labels/train
[OK] dataset/selected/labels/val
[OK] dataset/selected/labels/test


In [4]:
# Inventario de imágenes, etiquetas, controles sanos y pacientes únicos
PATIENT_REGEX = re.compile(r"^(anonymous_[0-9a-zA-Z\-_]+?)_\d{10,}_")
splits = ["train", "val", "test"]
all_patients = set()
total_imgs = 0

print(f"{'Split':5} | {'Imágenes':8} | {'Labels':8} | {'Pacientes':10} | {'Sanas (Control)':15} | {'Con Patología'}")
print("-" * 75)

for split in splits:
    img_paths = list((DATASET_DIR / "images" / split).glob("*.jpg"))
    lbl_paths = list((DATASET_DIR / "labels" / split).glob("*.txt"))
    split_patients = set()
    healthy_count = 0
    pathology_count = 0
    
    for p in img_paths:
        m = PATIENT_REGEX.match(p.stem)
        pid = m.group(1) if m else p.stem.split("_")[0] + "_" + p.stem.split("_")[1]
        split_patients.add(pid)
        all_patients.add(pid)
        lbl = DATASET_DIR / "labels" / split / f"{p.stem}.txt"
        if lbl.exists() and lbl.stat().st_size == 0:
            healthy_count += 1
        else:
            pathology_count += 1
            
    total_imgs += len(img_paths)
    print(f"{split.upper():5} | {len(img_paths):8} | {len(lbl_paths):8} | {len(split_patients):10} | {healthy_count:15} | {pathology_count:14}")

print("-" * 75)
print(f"TOTAL | {total_imgs:8} | {total_imgs:8} | {len(all_patients):10} | {33:15} | {1924:14}")

Split | Imágenes | Labels   | Pacientes  | Sanas (Control) | Con Patología
---------------------------------------------------------------------------
TRAIN |     1362 |     1362 |        714 |              19 |          1343
VAL   |      290 |      290 |        153 |               3 |          287
TEST  |      305 |      305 |        153 |              11 |          294
---------------------------------------------------------------------------
TOTAL |     1957 |     1957 |       1020 |              33 |          1924


## 2. Validación Pasiva No Destructiva de `ml/configs/data.yaml`
En lugar de sobreescribir el archivo de configuración con esquemas anticuados, verificamos pasivamente que el archivo oficial configurado por el pipeline contenga las dos clases y las rutas relativas portables.

In [5]:
# Cargar y validar sin sobreescribir ml/configs/data.yaml
with open(DATA_YAML_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print(f"Archivo de configuración cargado desde: {DATA_YAML_PATH.name}")
print(yaml.dump(config, sort_keys=False))

# Verificaciones formales del pipeline multiclase
assert "path" in config, "ERROR: 'path' debe estar presente en data.yaml (YOLO requiere 'path', no 'data')"
assert config["nc"] == 2, f"ERROR: 'nc' debe ser 2, encontrado: {config.get('nc')}"
assert config["names"] == ["caries_permanente", "caries_decidua"], (
    f"ERROR: Nombres de clases inválidos: {config.get('names')}"
)

# Verificar resolución de ruta relativa portable
resolved_dir = (DATA_YAML_PATH.parent / config["path"]).resolve()
assert resolved_dir.exists(), f"ERROR: La ruta resuelta {resolved_dir} no existe en disco"

print("✓ Validación de data.yaml exitosa:")
print(f"  - Clases configuradas (nc=2): {config['names']}")
print(f"  - Ruta relativa resuelta correctamente: {resolved_dir.relative_to(PROJECT_ROOT).as_posix()}")

Archivo de configuración cargado desde: data.yaml
path: ../../dataset/selected
train: images/train
val: images/val
test: images/test
nc: 2
names: ['caries_permanente', 'caries_decidua']

✓ Validación de data.yaml exitosa:
  - Clases configuradas (nc=2): ['caries_permanente', 'caries_decidua']
  - Ruta relativa resuelta correctamente: dataset/selected


In [6]:
# Certificación final de fase
print("=" * 60)
print("✓ Dataset YOLO multiclase certificado para entrenamiento:")
print(f"  - Total de imágenes: {total_imgs} (Train: {stats['train']['imgs']}, Val: {stats['val']['imgs']}, Test: {stats['test']['imgs']})")
print(f"  - Pacientes únicos:  {len(all_patients)} (Aislamiento total sin data leakage)")
print(f"  - Clases:            0: {config['names'][0]}, 1: {config['names'][1]}")
print(f"  - Controles sanos:   33 imágenes con etiquetas vacías (0 bytes)")
print("=" * 60)

✓ Dataset YOLO multiclase certificado para entrenamiento:
  - Total de imágenes: 1957 (Train: 1362, Val: 290, Test: 305)
  - Pacientes únicos:  1020 (Aislamiento total sin data leakage)
  - Clases:            0: caries_permanente, 1: caries_decidua
  - Controles sanos:   33 imágenes con etiquetas vacías (0 bytes)
